In [0]:
This is data warehouse pipeline practical project by converts raw retail data into trusted, analytics-ready insights through structured layers, enabling better and faster business decisions.

# Bronze Layer
- Import Required Library
- Define schema
- Data Collection
- Data Preprocessing i.e Data Injection

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

1.Define shema as string

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

2. Data Collection

- Read Raw Data As csv

In [0]:
orders_bronze_df = spark.read.format('csv').option("header", "true").schema(orders_schema).load("/Volumes/re_project/raw_data/raw_file/retail_orders_messy.csv")

customers_bronze_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(customer_schema) \
    .load("/Volumes/re_project/raw_data/raw_file/retail_customers_messy.csv")

In [0]:
orders_bronze_df.display()
customers_bronze_df.display()

Data Preprocessing

- Add ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp


orders_bronze_df = orders_bronze_df.withColumn('ingest_timestamp', current_timestamp())
customers_bronze_df = customers_bronze_df.withColumn('ingest_timestamp',current_timestamp())

In [0]:
orders_bronze_df.display()
customers_bronze_df.display()

- Save AS Delta Table in Bronze

In [0]:
orders_bronze_df.write.format("delta").mode("overwrite").saveAsTable('re_project.bronze.bronze_orders')
customers_bronze_df.write.format("delta").mode("overwrite").saveAsTable('re_project.bronze.bronze_customers')

# Silver Layer Transformation

SILVER LAYER OBJECTIVES

we will clean and transform the retail orders and customer dataset by:

- Fixing date formats

- Standardizing category values

- Removing duplicates

- Handling null values

- Converting discount from "5%" to 0.05 and "10%" to 0.1

- Removing negative quantities

- Fixing casing issues

- Casting unit_price to decimal

- Validating emails

- Recalculating total_amount

-- and more

Step 1: Load Bronze Orders

In [0]:

bronze_order_df = spark.read.table("re_project.bronze.bronze_orders")

display(bronze_order_df)

Step 2: Fix Date Formats

- Convert the Date column datatype to right datatype i.e Date columns

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
    coalesce(
        try_to_date(col('order_date'),'yyyy-MM-dd'),
        try_to_date(col('order_date'),'dd-MM-yyyy'),
        try_to_date(col('order_date'),'yyyy/MM/dd'),
        try_to_date(col('order_date'),'dd/MM/yyyy')
    )
)
silver_df.display()

Step 3: Standardize Category Values

- Display the unique value in the category columns

In [0]:
silver_df.select('category').distinct().display()

- Triming the wide spaces in the category value

In [0]:
from pyspark.sql.functions import upper, trim,when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

In [0]:
silver_df.select('category').distinct().display()

Step 4 Remove Duplicate Orders

- check the number of duplicate we have

In [0]:
from pyspark.sql.functions import count ,countDistinct

silver_df.select(count('order_id'),countDistinct('order_id')).display()

- Let drop the duplicate and also confirm it

In [0]:

silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'),countDistinct('order_id')).display()

- Clean the discount columns
  - let handle and replace the Null value and replace it with "0"

In [0]:
from pyspark.sql.functions import when, col

silver_df = silver_df.withColumn('discount',
                                 when(col('discount').isNull(),'0').when(col('discount')=='ten','0.1').otherwise(col('discount'))
                                 
                                 )

In [0]:

silver_df.display()

 Step 6: Convert Discount from "5%" to Decimal
some rows contain:

5%

we need 0.05

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn(
    "discount",
    when(col("discount").like("5%"), 
           (regexp_replace("discount", "5%", "0.05"))
    ).when(col("discount").like("10%"), 
           (regexp_replace("discount", "10%", "0.1"))
    ).otherwise(col("discount"))
)

silver_df.display()

- Step 7 Remove Negative Quantity

In [0]:
silver_df = silver_df.filter(col("quantity")>=0)

In [0]:
silver_df.display()

- Step 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_df.display()

Step 9: Recalculate Total Amount

In [0]:

silver_df = silver_df.withColumn(
    "total_amount",
    col("quantity").cast("int") * col("unit_price") * (1 - col("discount").cast("double"))
)

In [0]:

silver_df.display()

### Let Save the silver transform silver table

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("re_project.silver.silver_orders")

Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project

#### Customer_bronze

- Step 1: Read Bronze Table

In [0]:
customers_bronze_df = spark.table("re_project.bronze.bronze_customers")

display(customers_bronze_df)

- Step 2: Triminhg all the columns wide Spaces

In [0]:
from pyspark.sql.functions import trim, col

customers_df = customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_timestamp")
)

- Step 3: Remove Duplicate Customers

In [0]:

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("customer_id") \
                    .orderBy(desc("ingest_timestamp"))

customers_df = customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

customers_df = customers_df.filter("row_num = 1") \
                           .drop("row_num")

customers_df.display()

- Step 4: Fix and Standardize Status
  - From lower  letter to upper Letter

In [0]:
from pyspark.sql.functions import upper, when

customers_df = customers_df.withColumn(
    "status",
    upper(col("status"))
)

customers_df = customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)

customers_df.display()

- Step 5: Convert loyalty_points to Integer
  - Not Integer value to Null Value

In [0]:

from pyspark.sql.functions import regexp_replace

customers_df = customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), "[^0-9]", "")
)

customers_df = customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").try_cast("int")
)

In [0]:
customers_df.display()

Step 6: Validate Email Format

In [0]:
from pyspark.sql.functions import col

customers_df = customers_df.withColumn(
    "email_valid",
    col("email").rlike(r"^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

# Example:
# meenadas@email.com  -> valid
# karansingh@email    -> invalid

customers_df = customers_df.filter(col("email_valid") == True)

# customers_df.select("email", "email_valid").display()

- Step 7: Fixing the [](url) Date Formats by convert date of bith and registration date and dropping the old date of birth & registrations

In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date

customers_df = customers_df.withColumn(
    "dob_parsed",
    coalesce(
        try_to_date(col("date_of_birth"), "yyyy-MM-dd"),
        try_to_date(col("date_of_birth"), "dd-MM-yyyy"),
        try_to_date(col("date_of_birth"), "yyyy/MM/dd"),
        try_to_date(col("date_of_birth"), "dd/MM/yyyy")
    )
)

customers_df = customers_df.withColumn(
    "registration_parsed",
    coalesce(
        try_to_date(col("registration_date"), "yyyy-MM-dd"),
        try_to_date(col("registration_date"), "dd-MM-yyyy"),
        try_to_date(col("registration_date"), "yyyy/MM/dd"),
        try_to_date(col("registration_date"), "dd/MM/yyyy")
    )
)


customers_df.display()

In [0]:

customers_df = customers_df.drop("date_of_birth", "registration_date") \
                           .withColumnRenamed("dob_parsed", "date_of_birth") \
                           .withColumnRenamed("registration_parsed", "registration_date")

In [0]:
customers_df.display()

- Step 8: Standardize Gender columns

In [0]:
from pyspark.sql.functions import col, upper, when

customers_df = customers_df.withColumn(
    "gender",
    upper(col("gender"))
)

customers_df = customers_df.withColumn(
    "gender",
    when(col("gender") == "MALE", "M")
    .when(col("gender") == "FEMALE", "F")
    .otherwise("U")
)

customers_df.display()

##### Save to silver as silver_customers

In [0]:

customers_df.write.format("delta").mode("overwrite").saveAsTable("re_project.silver.silver_customers")

####  Build Gold Layer - Building Facts & Dimention Table(Retail_End_to_End Project)

 - Create Fact Table
 - Create Dimention Table
 - Join Orders & Customers Table
 - Build aggregated KPI
 - Store Optimize Gold table

Step 1: Load Silver Tables

In [0]:
customers_df = spark.read.table("re_project.silver.silver_customers")
orders_df = spark.read.table("re_project.silver.silver_orders")

customers_df.display()
orders_df.display()

Step 2: Create Customer Dimension

In [0]:
dim_customer = customers_df.select("customer_id",
                    "customer_name",
                    "email",
                    "city",
                    "state",
                    "loyalty_points",
                    "status"                    
                    )
dim_customer.display()

### Save it to Gold Layer

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

# fact_sales.display() by selected New reqirement columns


from pyspark.sql.functions import year , month , dayofmonth

order_enriched = fact_sales.\
    withColumn("Year",year("order_date")).\
    withColumn("Month",month("order_date")).\
    withColumn("Day",dayofmonth("order_date"))

order_enriched.display()


Let Aggregated average minimum and maximun function

In [0]:
from pyspark.sql.functions import sum, countDistinct, avg, min,max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()

In [0]:
gold_df.write.format('delta').mode('overwrite').saveAsTable('re_project.gold.gold_fact_Sales')

#### Sales Dashboard link below:

In [0]:
https://dbc-ebf7b20c-130a.cloud.databricks.com/dashboardsv3/01f126d17a631f32b390edab43e2ff78/published?o=7474653970713221